In [3]:
%pip install -q --upgrade --force-reinstall numpy==2.2.6 pandas==2.2.3 scipy==1.15.3 scikit-learn==1.7.2 bottleneck==1.5.0 gradio plotly

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-cloud-bigquery 3.46.2 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 3.0.0 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 3.0.0 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
holidays 0.107 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 3.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-

In [2]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import gradio as gr
import plotly

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Gradio:", gr.__version__)
print("All required imports successful!")

NumPy: 2.2.6
Pandas: 2.2.3
SciPy: 1.15.3
Scikit-learn: 1.7.2
Gradio: 6.30.0
All required imports successful!


In [3]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score
)

np.random.seed(42)
N = 1800

# Infosys-themed portfolio project.
# All employee records are synthetic.

departments = [
    "Technology", "Sales", "HR",
    "Finance", "Operations", "Customer Support"
]

roles = [
    "Associate", "Senior Associate",
    "Manager", "Senior Manager", "Consultant"
]

education = ["Bachelor's", "Master's", "Other"]

df = pd.DataFrame({
    "Employee_ID": [f"EMP{i:04d}" for i in range(1, N + 1)],
    "Department": np.random.choice(departments, N),
    "Job_Role": np.random.choice(roles, N),
    "Education": np.random.choice(education, N),
    "Age": np.random.randint(21, 61, N),
    "Tenure_Years": np.random.randint(0, 21, N),
    "Monthly_Salary": np.random.randint(25000, 160001, N),
    "Overtime_Hours": np.random.randint(0, 41, N),
    "Absence_Days": np.random.randint(0, 16, N),
    "Engagement_Score": np.random.randint(1, 11, N),
    "Job_Satisfaction": np.random.randint(1, 6, N),
    "Performance_Rating": np.random.randint(1, 6, N),
    "Training_Hours": np.random.randint(0, 61, N),
    "Manager_Support": np.random.randint(1, 11, N),
    "Years_Since_Promotion": np.random.randint(0, 11, N),
    "Work_Life_Balance": np.random.randint(1, 6, N)
})

# Generate illustrative attrition labels from simulated risk factors.
risk_score = (
    0.07 * df["Overtime_Hours"]
    + 0.12 * df["Absence_Days"]
    + 0.65 * (10 - df["Engagement_Score"])
    + 0.55 * (5 - df["Job_Satisfaction"])
    + 0.45 * (10 - df["Manager_Support"])
    + 0.28 * df["Years_Since_Promotion"]
    + 0.55 * (5 - df["Work_Life_Balance"])
    - 0.035 * df["Training_Hours"]
    - 0.025 * df["Tenure_Years"]
)

probability = 1 / (1 + np.exp(-(risk_score - 5.0) / 2.6))

df["Attrition"] = np.where(
    np.random.random(N) < probability, "Yes", "No"
)

features = [
    "Department", "Job_Role", "Education", "Age",
    "Tenure_Years", "Monthly_Salary", "Overtime_Hours",
    "Absence_Days", "Engagement_Score", "Job_Satisfaction",
    "Performance_Rating", "Training_Hours", "Manager_Support",
    "Years_Since_Promotion", "Work_Life_Balance"
]

X = df[features]
y = (df["Attrition"] == "Yes").astype(int)

categorical_features = ["Department", "Job_Role", "Education"]
numeric_features = [
    column for column in features
    if column not in categorical_features
]

preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    ("numeric", StandardScaler(), numeric_features)
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(
        y_test, predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, predictions, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, probabilities)
}

print("Dataset created successfully:", df.shape)
print("Random Forest model trained successfully.")
print("\nMODEL EVALUATION")

for name, value in metrics.items():
    print(f"{name}: {value:.3f}")

print(
    f"\nSynthetic attrition rate: "
    f"{(df['Attrition'] == 'Yes').mean():.1%}"
)


Dataset created successfully: (1800, 17)
Random Forest model trained successfully.

MODEL EVALUATION
Accuracy: 0.789
Precision: 0.836
Recall: 0.917
ROC-AUC: 0.729

Synthetic attrition rate: 80.4%


In [4]:
print("Dataset:", "df" in globals())
print("Trained model:", "model" in globals())
print("Metrics:", "metrics" in globals())


Dataset: True
Trained model: True
Metrics: True


In [ ]:

# ============================================================
# INFOSYS WORKFORCE INTELLIGENCE
# Corporate dashboard — navy, ivory, teal and copper
# Run Cell 2 first. Uses synthetic data only.
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ---------------------- COLOR PALETTE -----------------------

NAVY = "#173452"
NAVY_DARK = "#122941"
NAVY_LIGHT = "#254D73"
COPPER = "#C77F55"
COPPER_LIGHT = "#E4B08B"
TEAL = "#347F7D"
TEAL_LIGHT = "#6CA6A0"
IVORY = "#F7F5F0"
WHITE = "#FFFFFF"
TEXT = "#183653"
MUTED = "#55708C"
BORDER = "#DCE3E8"
PALE = "#F5EAE0"
BLUE = "#3977AC"

# ---------------------- PLOT STYLE --------------------------

def polish(fig, height=360):
    fig.update_layout(
        height=height,
        paper_bgcolor=WHITE,
        plot_bgcolor=WHITE,
        font=dict(family="Arial, sans-serif", size=12, color=NAVY),
        title=dict(
            font=dict(size=16, color=NAVY),
            x=0.035,
            xanchor="left"
        ),
        margin=dict(l=24, r=32, t=62, b=45),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.01,
            x=0.02,
            font=dict(color=NAVY, size=11),
            bgcolor="rgba(255,255,255,0)"
        ),
        hoverlabel=dict(
            bgcolor=NAVY,
            font=dict(color=WHITE)
        )
    )
    fig.update_xaxes(
        showgrid=False,
        linecolor=BORDER,
        tickfont=dict(color=NAVY, size=10),
        automargin=True
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor="#E9EDF1",
        zeroline=False,
        tickfont=dict(color=NAVY, size=10),
        automargin=True
    )
    return fig

# ---------------------- HEADER AND KPI CARDS ----------------

total_employees = len(df)
attrition_count = int((df["Attrition"] == "Yes").sum())
retained_count = total_employees - attrition_count
attrition_rate = attrition_count / total_employees * 100



def make_kpis():
    cards = [
        ("EMPLOYEE RECORDS", f"{total_employees:,}",
         "Synthetic demonstration dataset", TEAL, "●"),
        ("SIMULATED ATTRITION", f"{attrition_rate:.1f}%",
         f"{attrition_count:,} illustrative records", COPPER, "↗"),
        ("RETAINED EMPLOYEES", f"{retained_count:,}",
         "Records labelled as retained", TEAL, "✓"),
        ("MODEL ROC-AUC", f"{metrics['ROC-AUC']:.3f}",
         "Synthetic holdout evaluation", NAVY, "◎")
    ]

    html = '<div class="kpi-grid">'

    for label, value, note, accent, symbol in cards:
        html += f"""
        <div class="kpi-card">
            <div class="kpi-icon" style="background:{accent}">
                {symbol}
            </div>
            <div class="kpi-content">
                <div class="kpi-label">{label}</div>
                <div class="kpi-value">{value}</div>
                <div class="kpi-note">{note}</div>
            </div>
        </div>
        """

    html += "</div>"
    return html

# ---------------------- OVERVIEW CHARTS ---------------------

dept_summary = (
    df.groupby("Department")
    .agg(
        Employees=("Employee_ID", "count"),
        Attrition_Rate=(
            "Attrition",
            lambda s: (s == "Yes").mean() * 100
        )
    )
    .reset_index()
    .sort_values("Attrition_Rate")
)

fig_department = px.bar(
    dept_summary,
    x="Attrition_Rate",
    y="Department",
    orientation="h",
    text="Attrition_Rate",
    title="Simulated Attrition Rate by Department",
    color_discrete_sequence=[COPPER]
)
fig_department.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False
)
fig_department.update_layout(
    xaxis_title="Attrition rate (%)",
    yaxis_title="",
    xaxis_range=[0, min(100, dept_summary["Attrition_Rate"].max() + 15)]
)
fig_department = polish(fig_department, 330)

fig_engagement = px.histogram(
    df,
    x="Engagement_Score",
    color="Attrition",
    barmode="group",
    title="Engagement Score Distribution",
    color_discrete_map={"Yes": COPPER, "No": TEAL}
)
fig_engagement.update_layout(
    xaxis_title="Engagement score (1–10)",
    yaxis_title="Number of records"
)
fig_engagement = polish(fig_engagement, 330)

fig_overtime = px.box(
    df,
    x="Attrition",
    y="Overtime_Hours",
    color="Attrition",
    points=False,
    title="Overtime Hours by Attrition Label",
    color_discrete_map={"Yes": COPPER, "No": TEAL}
)
fig_overtime.update_layout(
    xaxis_title="Simulated attrition label",
    yaxis_title="Overtime hours"
)
fig_overtime = polish(fig_overtime, 330)

# ---------------------- ROOT-CAUSE EXPLORER -----------------

driver_options = {
    "Overtime Hours": "Overtime_Hours",
    "Absence Days": "Absence_Days",
    "Engagement Score": "Engagement_Score",
    "Job Satisfaction": "Job_Satisfaction",
    "Manager Support": "Manager_Support",
    "Training Hours": "Training_Hours",
    "Years Since Promotion": "Years_Since_Promotion",
    "Work-Life Balance": "Work_Life_Balance",
    "Tenure": "Tenure_Years",
    "Monthly Salary": "Monthly_Salary",
    "Age": "Age",
    "Performance Rating": "Performance_Rating"
}

def explore_driver(driver_label, group_by):
    driver = driver_options[driver_label]

    summary = (
        df.groupby(group_by, observed=True)
        .agg(
            Employees=("Employee_ID", "count"),
            Attrition_Rate=(
                "Attrition",
                lambda s: (s == "Yes").mean() * 100
            ),
            Average_Driver=(driver, "mean")
        )
        .reset_index()
        .round(2)
        .sort_values("Attrition_Rate")
    )

    fig = px.bar(
        summary,
        x="Attrition_Rate",
        y=group_by,
        orientation="h",
        text="Attrition_Rate",
        title=f"Attrition Comparison by {group_by.replace('_', ' ')}",
        color_discrete_sequence=[COPPER]
    )
    fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        cliponaxis=False
    )
    fig.update_layout(
        xaxis_title="Simulated attrition rate (%)",
        yaxis_title=""
    )
    fig = polish(fig, 370)

    explanation = (
        f"**Factor selected:** {driver_label}\n\n"
        f"**Grouped by:** {group_by.replace('_', ' ')}\n\n"
        "This view compares patterns in simulated data. It does not "
        "establish that a factor causes attrition."
    )

    return fig, summary, explanation

# ---------------------- DEPARTMENT INTELLIGENCE -------------

def department_intelligence(department):
    subset = df[df["Department"] == department]

    summary = (
        subset.groupby("Job_Role")
        .agg(
            Employees=("Employee_ID", "count"),
            Attrition_Rate=(
                "Attrition",
                lambda s: (s == "Yes").mean() * 100
            ),
            Avg_Engagement=("Engagement_Score", "mean"),
            Avg_Satisfaction=("Job_Satisfaction", "mean"),
            Avg_Overtime=("Overtime_Hours", "mean")
        )
        .reset_index()
        .round(2)
        .sort_values("Attrition_Rate")
    )

    fig = px.bar(
        summary,
        x="Job_Role",
        y="Attrition_Rate",
        text="Attrition_Rate",
        title=f"{department}: Attrition by Job Role",
        color_discrete_sequence=[COPPER]
    )
    fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        cliponaxis=False
    )
    fig.update_layout(
        xaxis_title="Job role",
        yaxis_title="Simulated attrition rate (%)"
    )
    fig = polish(fig, 350)

    highest = summary.sort_values(
        "Attrition_Rate", ascending=False
    ).iloc[0]

    narrative = (
        f"### {department}\n\n"
        f"**Employee records:** {len(subset):,}\n\n"
        f"**Highest role-level rate in this sample:** "
        f"{highest['Job_Role']} ({highest['Attrition_Rate']:.1f}%).\n\n"
        "This is a descriptive result from synthetic records, not an "
        "actual company finding."
    )

    return fig, summary, narrative

# ---------------------- RETENTION STRATEGY AGENT ------------

def retention_recommendations():
    suggestions = []

    if df["Engagement_Score"].mean() < 6.5:
        suggestions.append((
            "Engagement",
            "Introduce regular engagement check-ins and follow up "
            "on employee feedback.",
            "TEAL"
        ))

    if df["Overtime_Hours"].mean() > 20:
        suggestions.append((
            "Workload balance",
            "Review workload allocation, staffing levels and recurring "
            "overtime to identify preventable pressure.",
            "COPPER"
        ))

    if df["Years_Since_Promotion"].mean() > 4:
        suggestions.append((
            "Career growth",
            "Review internal mobility, career pathways and promotion "
            "criteria with employees.",
            "BLUE"
        ))

    if df["Manager_Support"].mean() < 6.5:
        suggestions.append((
            "Manager support",
            "Strengthen manager coaching and regular one-to-one "
            "conversations.",
            "TEAL"
        ))

    if df["Training_Hours"].mean() < 25:
        suggestions.append((
            "Learning and development",
            "Review access to role-relevant training and opportunities "
            "to develop new skills.",
            "COPPER"
        ))

    if not suggestions:
        suggestions.append((
            "Ongoing support",
            "Continue monitoring engagement, workload, career growth "
            "and employee feedback.",
            "TEAL"
        ))

    summary = (
        df.groupby("Department")
        .agg(
            Employees=("Employee_ID", "count"),
            Attrition_Rate=(
                "Attrition",
                lambda s: (s == "Yes").mean() * 100
            ),
            Avg_Engagement=("Engagement_Score", "mean"),
            Avg_Overtime=("Overtime_Hours", "mean"),
            Avg_Satisfaction=("Job_Satisfaction", "mean")
        )
        .reset_index()
        .round(2)
        .sort_values("Attrition_Rate", ascending=False)
    )

    cards = ""
    for title, description, color_name in suggestions:
        color = {
            "TEAL": TEAL,
            "COPPER": COPPER,
            "BLUE": BLUE
        }[color_name]

        cards += f"""
        <div class="strategy-card">
            <div class="strategy-accent" style="background:{color}"></div>
            <div class="strategy-body">
                <div class="strategy-title">{title}</div>
                <div class="strategy-description">{description}</div>
            </div>
        </div>
        """

    top_departments = ""
    for _, row in summary.head(3).iterrows():
        top_departments += f"""
        <tr>
            <td>{row['Department']}</td>
            <td>{int(row['Employees']):,}</td>
            <td>{row['Attrition_Rate']:.1f}%</td>
            <td>{row['Avg_Engagement']:.1f}/10</td>
            <td>{row['Avg_Overtime']:.1f} hrs</td>
        </tr>
        """

    strategy_html = f"""
    <div class="strategy-intro">
        <div class="strategy-eyebrow">PEOPLE & CULTURE</div>
        <div class="strategy-heading">Retention Strategy Planner</div>
        <div class="strategy-subtitle">
            Practical, supportive actions based on patterns in the
            synthetic demonstration dataset.
        </div>
    </div>

    <div class="strategy-section-title">Recommended focus areas</div>
    <div class="strategy-grid">
        {cards}
    </div>

    <div class="strategy-section-title">Departments to review first</div>
    <div class="strategy-table-wrap">
        <table class="strategy-table">
            <thead>
                <tr>
                    <th>Department</th>
                    <th>Records</th>
                    <th>Attrition</th>
                    <th>Engagement</th>
                    <th>Overtime</th>
                </tr>
            </thead>
            <tbody>{top_departments}</tbody>
        </table>
    </div>

    <div class="strategy-callout">
        <strong>Suggested next step</strong>
        <p>
        Validate these patterns with confidential employee feedback
        and workload reviews before selecting an intervention.
        </p>
    </div>

    <div class="strategy-disclaimer">
        These recommendations are generated by transparent rules,
        not an LLM. All figures are synthetic and do not describe
        actual Infosys employees.
    </div>
    """

    return strategy_html, summary

strategy_html, recommendation_table = retention_recommendations()

# ---------------------- EMPLOYEE RISK ANALYZER ---------------

def analyze_employee(
    department, role, education_level, age, tenure, salary,
    overtime, absences, engagement, satisfaction, performance,
    training, manager_support, years_since_promotion, work_life_balance
):
    record = pd.DataFrame([{
        "Department": department,
        "Job_Role": role,
        "Education": education_level,
        "Age": int(age),
        "Tenure_Years": int(tenure),
        "Monthly_Salary": float(salary),
        "Overtime_Hours": int(overtime),
        "Absence_Days": int(absences),
        "Engagement_Score": int(engagement),
        "Job_Satisfaction": int(satisfaction),
        "Performance_Rating": int(performance),
        "Training_Hours": int(training),
        "Manager_Support": int(manager_support),
        "Years_Since_Promotion": int(years_since_promotion),
        "Work_Life_Balance": int(work_life_balance)
    }])

    risk = float(model.predict_proba(record)[0, 1]) * 100

    if risk >= 65:
        band = "Higher model-estimated risk"
        action = "Offer a supportive check-in about workload, career growth and workplace support."
    elif risk >= 35:
        band = "Moderate model-estimated risk"
        action = "Review workload balance, engagement and development opportunities."
    else:
        band = "Lower model-estimated risk"
        action = "Continue regular supportive check-ins."

    result = f"""
## Illustrative model result

**Estimated probability:** {risk:.1f}%

**Model risk band:** {band}

**Suggested supportive action:** {action}

This result is generated from a model trained on synthetic data.
It is not a validated employee assessment and must not be used
to make employment decisions.
"""

    classifier = model.named_steps["classifier"]
    names = model.named_steps["preprocessing"].get_feature_names_out()

    importance = pd.DataFrame({
        "Feature": names,
        "Importance": classifier.feature_importances_
    }).nlargest(10, "Importance")

    importance["Feature"] = (
        importance["Feature"]
        .str.replace("numeric__", "", regex=False)
        .str.replace("categorical__", "", regex=False)
    )

    fig = px.bar(
        importance.sort_values("Importance"),
        x="Importance",
        y="Feature",
        orientation="h",
        title="Global Model Feature Importance",
        color_discrete_sequence=[COPPER]
    )
    fig.update_layout(xaxis_title="Relative importance", yaxis_title="")
    fig = polish(fig, 370)

    return result, fig

# ---------------------- EMPLOYEE EXPLORER --------------------

def filter_employees(department, attrition_status, min_engagement, max_overtime):
    result = df.copy()

    if department != "All":
        result = result[result["Department"] == department]
    if attrition_status != "All":
        result = result[result["Attrition"] == attrition_status]

    result = result[
        (result["Engagement_Score"] >= int(min_engagement))
        & (result["Overtime_Hours"] <= int(max_overtime))
    ]

    columns = [
        "Employee_ID", "Department", "Job_Role",
        "Tenure_Years", "Engagement_Score", "Job_Satisfaction",
        "Overtime_Hours", "Manager_Support", "Attrition"
    ]

    return (
        result[columns].head(250),
        f"**{len(result):,} records match your filters.** "
        "Maximum 250 rows displayed."
    )

# ---------------------- METHODOLOGY --------------------------

methodology = f"""
## Model and methodology

### Dataset
- **Records:** {len(df):,} synthetic employee profiles.
- **Target:** simulated attrition label.
- **Features:** role, tenure, workload, engagement, satisfaction,
  training and workplace-support variables.

### Machine learning
- **Algorithm:** Random Forest Classifier.
- **Train/test split:** 80% / 20%.
- **Preprocessing:** one-hot encoding and numeric scaling.
- **Class weighting:** balanced.

### Holdout evaluation

| Metric | Score |
|---|---:|
| Accuracy | {metrics['Accuracy']:.3f} |
| Precision | {metrics['Precision']:.3f} |
| Recall | {metrics['Recall']:.3f} |
| ROC-AUC | {metrics['ROC-AUC']:.3f} |

### Limitations
The labels are generated from simulated rules. These metrics do not
demonstrate predictive performance on actual employees.

The retention recommendations use simple rules and are not generated
by a large language model.

### Responsible use
This prototype is for portfolio demonstration only. It should not be
used for real hiring, firing, promotion or other employment decisions.
"""

# ---------------------- COMPLETE CSS -------------------------

custom_css = """
.gradio-container {
    background: #F7F5F0 !important;
    color: #183653 !important;
    font-family: Arial, Helvetica, sans-serif !important;
    max-width: 1500px !important;
}

/* Brand header */
#brand-bar {
    background: #173452 !important;
    color: #FFFFFF !important;
    margin: 0 0 14px 0 !important;
    padding: 18px 28px !important;
    border-radius: 10px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: space-between !important;
    gap: 20px !important;
}
#brand-bar .brand-left {
    display: flex !important;
    align-items: center !important;
    gap: 18px !important;
}
#brand-bar .brand-name {
    font-size: 28px !important;
    font-weight: 500 !important;
    letter-spacing: -1px !important;
    color: #FFFFFF !important;
}
#brand-bar .brand-divider {
    width: 1px;
    height: 35px;
    background: #A9BBD0;
}
#brand-bar .brand-product {
    font-size: 20px !important;
    font-weight: 700 !important;
    color: #FFFFFF !important;
}
#brand-bar .brand-tagline {
    color: #F7F5F0 !important;
    font-size: 11px !important;
    letter-spacing: .3px !important;
}

/* Page heading */
#main-title {
    background: #FFFFFF !important;
    border: 1px solid #DCE3E8 !important;
    border-radius: 11px !important;
    padding: 23px 26px !important;
    margin: 12px 0 18px 0 !important;
}
#main-title h1 {
    color: #173452 !important;
    font-size: 30px !important;
    font-weight: 750 !important;
    margin: 0 0 7px 0 !important;
    line-height: 1.3 !important;
}
#main-title p {
    color: #55708C !important;
    font-size: 14px !important;
    margin: 0 !important;
    line-height: 1.6 !important;
}
#main-title .portfolio-note {
    background: #F7F0E9 !important;
    border: 1px solid #E9D8C8 !important;
    border-radius: 8px !important;
    padding: 12px 15px !important;
    margin-top: 14px !important;
    color: #49382D !important;
    font-size: 12px !important;
    line-height: 1.6 !important;
}
#main-title .portfolio-note strong {
    color: #9B5B35 !important;
}

/* Navigation: visible at rest */
.tab-nav {
    display: flex !important;
    gap: 4px !important;
    padding: 6px !important;
    background: #FFFFFF !important;
    border: 1px solid #DCE3E8 !important;
    border-radius: 10px !important;
    margin-bottom: 18px !important;
}
.tab-nav button,
.tab-nav button span,
.tab-nav button p {
    color: #173452 !important;
    opacity: 1 !important;
    font-size: 12px !important;
    font-weight: 650 !important;
}
.tab-nav button {
    background: #FFFFFF !important;
    border: 1px solid transparent !important;
    border-radius: 7px !important;
    padding: 12px 10px !important;
    white-space: nowrap !important;
}
.tab-nav button:hover {
    background: #F5EAE0 !important;
    color: #173452 !important;
}
.tab-nav button[aria-selected="true"],
.tab-nav button.selected {
    background: #173452 !important;
    color: #FFFFFF !important;
    border-color: #173452 !important;
}
.tab-nav button[aria-selected="true"] *,
.tab-nav button.selected * {
    color: #FFFFFF !important;
}

/* KPI layout */
.kpi-grid {
    display: grid !important;
    grid-template-columns: repeat(4, minmax(0, 1fr)) !important;
    gap: 14px !important;
    width: 100% !important;
    margin: 12px 0 20px 0 !important;
}
.kpi-card {
    background: #FFFFFF !important;
    border: 1px solid #DCE3E8 !important;
    border-radius: 10px !important;
    padding: 17px !important;
    min-height: 118px !important;
    display: flex !important;
    align-items: flex-start !important;
    gap: 13px !important;
    box-shadow: 0 2px 7px rgba(23,52,82,.04) !important;
    min-width: 0 !important;
}
.kpi-icon {
    width: 46px !important;
    height: 46px !important;
    min-width: 46px !important;
    border-radius: 50% !important;
    color: #FFFFFF !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    font-size: 25px !important;
    font-weight: 700 !important;
}
.kpi-content {
    min-width: 0 !important;
    padding-top: 3px !important;
}
.kpi-label {
    color: #173452 !important;
    font-size: 10px !important;
    font-weight: 750 !important;
    letter-spacing: .55px !important;
    line-height: 1.5 !important;
}
.kpi-value {
    color: #173452 !important;
    font-size: 29px !important;
    font-weight: 750 !important;
    line-height: 1.25 !important;
    margin: 7px 0 !important;
    white-space: nowrap !important;
}
.kpi-note {
    color: #55708C !important;
    font-size: 10px !important;
    line-height: 1.45 !important;
}

/* Retention strategy section */
.strategy-intro {
    background: #173452 !important;
    border-radius: 12px !important;
    padding: 25px 27px !important;
    margin: 5px 0 24px 0 !important;
}
.strategy-eyebrow {
    color: #E4B08B !important;
    font-size: 10px !important;
    letter-spacing: 1.8px !important;
    font-weight: 700 !important;
    margin-bottom: 8px !important;
}
.strategy-heading {
    color: #FFFFFF !important;
    font-size: 28px !important;
    font-weight: 750 !important;
    line-height: 1.3 !important;
    margin-bottom: 8px !important;
}
.strategy-subtitle {
    color: #E1EAF2 !important;
    font-size: 13px !important;
    line-height: 1.7 !important;
}
.strategy-section-title {
    color: #173452 !important;
    font-size: 18px !important;
    font-weight: 750 !important;
    margin: 22px 0 13px 0 !important;
}
.strategy-grid {
    display: grid !important;
    grid-template-columns: repeat(2, minmax(0, 1fr)) !important;
    gap: 13px !important;
}
.strategy-card {
    display: flex !important;
    align-items: stretch !important;
    background: #FFFFFF !important;
    border: 1px solid #DCE3E8 !important;
    border-radius: 10px !important;
    overflow: hidden !important;
    min-height: 105px !important;
}
.strategy-accent {
    width: 5px !important;
    min-width: 5px !important;
}
.strategy-body {
    padding: 17px 18px !important;
}
.strategy-title {
    color: #173452 !important;
    font-size: 14px !important;
    font-weight: 750 !important;
    margin-bottom: 8px !important;
}
.strategy-description {
    color: #55708C !important;
    font-size: 12px !important;
    line-height: 1.65 !important;
}
.strategy-table-wrap {
    overflow-x: auto !important;
    background: #FFFFFF !important;
    border: 1px solid #DCE3E8 !important;
    border-radius: 10px !important;
    margin-bottom: 20px !important;
}
.strategy-table {
    width: 100% !important;
    border-collapse: collapse !important;
    background: #FFFFFF !important;
}
.strategy-table th {
    background: #173452 !important;
    color: #FFFFFF !important;
    font-size: 11px !important;
    font-weight: 700 !important;
    text-align: left !important;
    padding: 13px 15px !important;
    white-space: nowrap !important;
}
.strategy-table td {
    background: #FFFFFF !important;
    color: #183653 !important;
    font-size: 12px !important;
    padding: 13px 15px !important;
    border-bottom: 1px solid #E9EDF1 !important;
    white-space: nowrap !important;
}
.strategy-table tr:nth-child(even) td {
    background: #F7F9FB !important;
}
.strategy-callout {
    background: #F7F0E9 !important;
    border: 1px solid #E9D8C8 !important;
    border-left: 5px solid #C77F55 !important;
    border-radius: 9px !important;
    padding: 17px 20px !important;
    margin: 18px 0 !important;
}
.strategy-callout strong {
    color: #9B5B35 !important;
    font-size: 13px !important;
}
.strategy-callout p {
    color: #49382D !important;
    font-size: 12px !important;
    line-height: 1.65 !important;
    margin: 6px 0 0 0 !important;
}
.strategy-disclaimer {
    color: #55708C !important;
    font-size: 11px !important;
    line-height: 1.7 !important;
    border-top: 1px solid #DCE3E8 !important;
    padding-top: 14px !important;
    margin-top: 14px !important;
}

/* Inputs, labels and text */
.gradio-container label,
.gradio-container .label-wrap,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container .prose {
    color: #183653 !important;
}
.gradio-container h2 {
    color: #173452 !important;
    font-size: 21px !important;
}
.gradio-container h3 {
    color: #173452 !important;
    font-size: 16px !important;
}
.gradio-container p,
.gradio-container li {
    line-height: 1.7 !important;
}
.gradio-container input,
.gradio-container textarea,
.gradio-container select {
    background: #FFFFFF !important;
    color: #183653 !important;
    border-color: #DCE3E8 !important;
}
.gradio-container input::placeholder,
.gradio-container textarea::placeholder {
    color: #657C93 !important;
}

/* Buttons */
.gradio-container button.primary {
    background: #173452 !important;
    color: #FFFFFF !important;
    border: 1px solid #173452 !important;
    border-radius: 7px !important;
    font-weight: 700 !important;
}
.gradio-container button.primary:hover {
    background: #254D73 !important;
    color: #FFFFFF !important;
}

/* Plot panels and dataframes */
.plot-container, .gradio-plot {
    background: #FFFFFF !important;
    border: 1px solid #DCE3E8 !important;
    border-radius: 10px !important;
    overflow: hidden !important;
}
table {
    background: #FFFFFF !important;
}
thead, th {
    background: #173452 !important;
    color: #FFFFFF !important;
}
td {
    color: #183653 !important;
}

/* Footer */
#dashboard-footer {
    text-align: center !important;
    color: #55708C !important;
    font-size: 10px !important;
    letter-spacing: 1px !important;
    padding: 22px 0 10px 0 !important;
    border-top: 1px solid #DCE3E8 !important;
    margin-top: 20px !important;
}

/* Responsive layout */
@media (max-width: 1000px) {
    .kpi-grid {
        grid-template-columns: repeat(2, minmax(0, 1fr)) !important;
    }
    .tab-nav {
        overflow-x: auto !important;
        flex-wrap: nowrap !important;
    }
    .tab-nav button {
        white-space: nowrap !important;
    }
}
@media (max-width: 620px) {
    #brand-bar {
        flex-direction: column !important;
        align-items: flex-start !important;
    }
    #brand-bar .brand-name {
        font-size: 23px !important;
    }
    #brand-bar .brand-product {
        font-size: 15px !important;
    }
    .kpi-grid {
        grid-template-columns: 1fr 1fr !important;
        gap: 8px !important;
    }
    .kpi-card {
        padding: 12px !important;
        gap: 8px !important;
    }
    .kpi-icon {
        width: 32px !important;
        height: 32px !important;
        min-width: 32px !important;
        font-size: 18px !important;
    }
    .kpi-value {
        font-size: 22px !important;
    }
    .strategy-grid {
        grid-template-columns: 1fr !important;
    }
    .strategy-heading {
        font-size: 23px !important;
    }
}
"""

# ---------------------- BUILD DASHBOARD ----------------------

with gr.Blocks(
    title="Infosys Workforce Intelligence",
    theme=gr.themes.Base(
        primary_hue="blue",
        secondary_hue="teal",
        neutral_hue="slate"
    ),
    css=custom_css
) as demo:

    gr.HTML("""
    <div id="brand-bar">
        <div class="brand-left">
            <div class="brand-name">Infosys</div>
            <div class="brand-divider"></div>
            <div class="brand-product">Workforce Intelligence</div>
        </div>
        <div class="brand-tagline">
            Better People &nbsp; | &nbsp; Better Business
            &nbsp; | &nbsp; A Brighter Future
        </div>
    </div>

    <div id="main-title">
        <h1>Infosys Workforce Intelligence</h1>
        <p>Employee Attrition Insights &nbsp; • &nbsp;
        Root-Cause Exploration &nbsp; • &nbsp; Retention Strategy</p>
        <div class="portfolio-note">
            <strong>Portfolio demonstration:</strong>
            This dashboard uses synthetic data and is not an official
            Infosys tool or a representation of actual Infosys employee information.
        </div>
    </div>
    """)

    with gr.Tabs():

        # 1. EXECUTIVE OVERVIEW
        with gr.Tab("Executive Overview"):
            gr.Markdown("## Workforce at a glance")
            gr.HTML(make_kpis())

            with gr.Row(equal_height=True):
                with gr.Column():
                    gr.Plot(value=fig_department, label="Department attrition")
                with gr.Column():
                    gr.Plot(value=fig_engagement, label="Engagement distribution")

            gr.Plot(value=fig_overtime, label="Overtime comparison")

            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Key insights & next steps")
                    gr.Markdown(
                        f"- Highest department attrition rate in this sample: "
                        f"**{dept_summary.iloc[-1]['Department']}** "
                        f"({dept_summary.iloc[-1]['Attrition_Rate']:.1f}%).\n"
                        "- Review engagement, workload balance and career growth.\n"
                        "- Validate patterns with employee feedback before action."
                    )
                with gr.Column():
                    gr.Markdown("### Model performance")
                    gr.Plot(
                        value=go.Figure(
                            data=[go.Pie(
                                labels=["ROC-AUC", "Remaining scale"],
                                values=[
                                    metrics["ROC-AUC"],
                                    max(0, 1 - metrics["ROC-AUC"])
                                ],
                                hole=0.68,
                                marker_colors=[NAVY, "#E8EDF2"],
                                textinfo="none",
                                hovertemplate="%{label}: %{value:.3f}<extra></extra>"
                            )],
                            layout=go.Layout(
                                height=260,
                                paper_bgcolor=WHITE,
                                plot_bgcolor=WHITE,
                                showlegend=False,
                                annotations=[dict(
                                    text=(
                                        f"<b>{metrics['ROC-AUC']:.3f}</b>"
                                        "<br><sup>ROC-AUC</sup>"
                                    ),
                                    x=0.5, y=0.5,
                                    font=dict(size=20, color=NAVY),
                                    showarrow=False
                                )],
                                margin=dict(l=10, r=10, t=15, b=15)
                            )
                        ),
                        label="Model performance"
                    )

        # 2. ROOT-CAUSE EXPLORER
        with gr.Tab("Root-Cause Explorer"):
            gr.Markdown("## Explore workforce patterns")
            gr.Markdown(
                "Choose a factor and compare simulated attrition rates "
                "across departments, job roles or education groups."
            )
            with gr.Row():
                driver_dropdown = gr.Dropdown(
                    choices=list(driver_options.keys()),
                    value="Engagement Score",
                    label="Workplace factor"
                )
                group_dropdown = gr.Dropdown(
                    choices=["Department", "Job_Role", "Education"],
                    value="Department",
                    label="Compare groups"
                )

            root_chart = gr.Plot()
            root_explanation = gr.Markdown()
            root_table = gr.Dataframe(
                label="Group summary",
                interactive=False
            )

            root_inputs = [driver_dropdown, group_dropdown]
            root_outputs = [root_chart, root_table, root_explanation]

            driver_dropdown.change(
                explore_driver, root_inputs, root_outputs
            )
            group_dropdown.change(
                explore_driver, root_inputs, root_outputs
            )
            demo.load(
                explore_driver, root_inputs, root_outputs
            )

        # 3. DEPARTMENT INTELLIGENCE
        with gr.Tab("Department Intelligence"):
            gr.Markdown("## Department-level intelligence")
            department_dropdown = gr.Dropdown(
                choices=departments,
                value=departments[0],
                label="Select department"
            )
            dept_chart = gr.Plot()
            dept_narrative = gr.Markdown()
            dept_table = gr.Dataframe(
                label="Job-role summary",
                interactive=False
            )

            department_dropdown.change(
                department_intelligence,
                [department_dropdown],
                [dept_chart, dept_table, dept_narrative]
            )
            demo.load(
                department_intelligence,
                [department_dropdown],
                [dept_chart, dept_table, dept_narrative]
            )

        # 4. RETENTION STRATEGY
        with gr.Tab("Retention Strategy"):
            gr.HTML(strategy_html)

            gr.Markdown("### Department-level summary")
            gr.Dataframe(
                value=recommendation_table,
                label="Department comparison",
                interactive=False
            )

        # 5. EMPLOYEE RISK ANALYZER
        with gr.Tab("Employee Risk Analyzer"):
            gr.Markdown("## Hypothetical employee profile")
            gr.Markdown(
                "Adjust the example inputs to explore the model output. "
                "Do not enter real employee information."
            )

            with gr.Row():
                with gr.Column():
                    risk_department = gr.Dropdown(
                        departments, value="Technology", label="Department"
                    )
                    risk_role = gr.Dropdown(
                        roles, value="Associate", label="Job role"
                    )
                    risk_education = gr.Dropdown(
                        education, value="Bachelor's", label="Education"
                    )
                    risk_age = gr.Slider(
                        21, 60, value=30, step=1, label="Age"
                    )
                    risk_tenure = gr.Slider(
                        0, 20, value=3, step=1, label="Tenure (years)"
                    )
                    risk_salary = gr.Number(
                        value=60000, label="Monthly salary (illustrative)"
                    )
                    risk_overtime = gr.Slider(
                        0, 40, value=10, step=1, label="Overtime hours"
                    )
                    risk_absences = gr.Slider(
                        0, 15, value=3, step=1, label="Absence days"
                    )

                with gr.Column():
                    risk_engagement = gr.Slider(
                        1, 10, value=7, step=1, label="Engagement (1–10)"
                    )
                    risk_satisfaction = gr.Slider(
                        1, 5, value=4, step=1, label="Job satisfaction (1–5)"
                    )
                    risk_performance = gr.Slider(
                        1, 5, value=3, step=1, label="Performance rating (1–5)"
                    )
                    risk_training = gr.Slider(
                        0, 60, value=20, step=1, label="Training hours"
                    )
                    risk_manager_support = gr.Slider(
                        1, 10, value=7, step=1, label="Manager support (1–10)"
                    )
                    risk_promotion = gr.Slider(
                        0, 10, value=2, step=1, label="Years since promotion"
                    )
                    risk_balance = gr.Slider(
                        1, 5, value=3, step=1, label="Work-life balance (1–5)"
                    )

            analyze_button = gr.Button(
                "Analyze Hypothetical Profile",
                variant="primary"
            )
            risk_result = gr.Markdown()
            importance_plot = gr.Plot()

            analyze_button.click(
                analyze_employee,
                inputs=[
                    risk_department, risk_role, risk_education,
                    risk_age, risk_tenure, risk_salary,
                    risk_overtime, risk_absences, risk_engagement,
                    risk_satisfaction, risk_performance, risk_training,
                    risk_manager_support, risk_promotion, risk_balance
                ],
                outputs=[risk_result, importance_plot]
            )

        # 6. EMPLOYEE EXPLORER
        with gr.Tab("Employee Explorer"):
            gr.Markdown("## Employee record explorer")
            gr.Markdown(
                "Filter synthetic records. The displayed employee IDs "
                "are generated for this portfolio demonstration."
            )

            with gr.Row():
                filter_department = gr.Dropdown(
                    choices=["All"] + departments,
                    value="All",
                    label="Department"
                )
                filter_attrition = gr.Dropdown(
                    choices=["All", "Yes", "No"],
                    value="All",
                    label="Attrition label"
                )

            with gr.Row():
                filter_engagement = gr.Slider(
                    1, 10, value=1, step=1,
                    label="Minimum engagement"
                )
                filter_overtime = gr.Slider(
                    0, 40, value=40, step=1,
                    label="Maximum overtime hours"
                )

            filter_button = gr.Button(
                "Apply Filters", variant="primary"
            )
            employee_count = gr.Markdown()
            employee_table = gr.Dataframe(
                label="Matching records",
                interactive=False
            )

            filter_inputs = [
                filter_department, filter_attrition,
                filter_engagement, filter_overtime
            ]
            filter_outputs = [employee_table, employee_count]

            filter_button.click(
                filter_employees, filter_inputs, filter_outputs
            )
            demo.load(
                filter_employees, filter_inputs, filter_outputs
            )

        # 7. METHODOLOGY
        with gr.Tab("Methodology"):
            gr.Markdown(methodology)
            gr.Dataframe(
                value=pd.DataFrame([
                    {"Metric": k, "Score": round(v, 3)}
                    for k, v in metrics.items()
                ]),
                label="Model evaluation",
                interactive=False
            )

    gr.HTML("""
    <div id="dashboard-footer">
        WORKFORCE INTELLIGENCE &nbsp; • &nbsp;
        PORTFOLIO PROJECT &nbsp; • &nbsp; SYNTHETIC DATA
    </div>
    """)

# ---------------------- LAUNCH ------------------------------

demo.launch(share=True, debug=True)


/tmp/ipykernel_4526/2887693749.py:1021: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://86e5c1f0c8c29dd45d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
